In [9]:
import pandas as pd

# Read the TSV file with specific columns
columns_to_read = ['rowid', 'Scan', 'Annotation','Score','ProtsAll', 'AnnotationOther', 'ChargeOther','ScoreOther','ProtsAllOther',
                   'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract']
mismatch_df = pd.read_csv('PRISMAL_REPROCESS_PA_UniproKB_Normal123_mismatched.tsv', sep='\t', usecols=columns_to_read)
matched_df = pd.read_csv('PRISMAL_REPROCESS_PA_UniproKB_Normal123_matched.tsv', sep='\t', usecols=columns_to_read)

# Display basic information about the dataset
print(f"Dataset shape: {mismatch_df.shape}")
print(f"Columns: {list(mismatch_df.columns)}")
print("\nFirst few rows:")
mismatch_df.head()

Dataset shape: (9020, 13)
Columns: ['rowid', 'Scan', 'Annotation', 'Score', 'ProtsAll', 'AnnotationOther', 'ChargeOther', 'ScoreOther', 'ProtsAllOther', 'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract']

First few rows:


,rowid,Scan,Annotation,Score,ProtsAll,AnnotationOther,ChargeOther,ScoreOther,ProtsAllOther,MinNTermAdd,minNTermSubtract,MinCTermAdd,minCTermSubtract
0,1,6352,+42.011FRNFGGLLGPMDEPVGMQKWGK,-25.036301,"(sp|Q86Y38|XYLT1_HUMAN,1,2,0,0)",KQVEVDAQQC+57.021MLEILDTAGTEQ,2,106.825996,"(sp|P61224|RAP1B_HUMAN,0,0,5,21);(tr|E7ESV4|E7...",0,0,5,21
1,2,6353,ADEEFQ+0.984ILANSWRYSSAFTNR,-23.241800,"(sp|Q9H0U3-2|MAGT1_HUMAN,1,13,0,0);(sp|Q9H0U3|...",KQVEVDAQQC+57.021MLEILDTAGTEQ,2,103.260002,"(sp|P61224|RAP1B_HUMAN,0,0,5,21);(tr|E7ESV4|E7...",0,0,5,21
2,3,21453,SGKEEALC+57.021QLQEENR,8.635220,"(sp|Q6DT37|MRCKG_HUMAN,1,3,0,0)",+42.011MM+15.995WSNFFLQEENR,2,95.855103,"(sp|Q9BXT2|CCG6_HUMAN,0,0,0,0);(tr|A6NFR2|A6NF...",0,0,0,0
3,4,6363,ADEEFQ+0.984ILANSWRYSSAFTNR,-22.113600,"(sp|Q9H0U3-2|MAGT1_HUMAN,1,13,0,0);(sp|Q9H0U3|...",KQVEVDAQQC+57.021MLEILDTAGTEQ,2,89.525597,"(sp|P61224|RAP1B_HUMAN,0,0,5,21);(tr|E7ESV4|E7...",0,0,5,21
4,5,5047,+43.006RSEYHEYSSSGPSGSR,-27.427601,"(XXX_sp|Q9P1Y6-2|PHRF1_HUMAN,0,0,0,0);(XXX_sp|...",DNGYEIYESENGEPRG,2,87.540901,"(sp|P21815|SIAL_HUMAN,18,15,4,1)",18,15,4,1


create a new tsv named mismatch_summary.tsv with the following columns: peptide, peptide_demod, reason_mismatch,'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract' where go through each column of mismatch_df, the peptide in new csv is  AnnotationOther, and peptide_demod is AnnotationOther but get rid of the modifications, for example "+42.011FRNF+28.011GGL" is FRNFGGLLGPMDEPVGMQKWGK. and 'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract' columns are the same from mismatch df. leave reason_mismatch empty for now

In [10]:
import re

def remove_modifications(peptide_string):
    """Remove modification annotations from peptide sequence"""
    if pd.isna(peptide_string):
        return ''
    # Remove modification patterns like +42.011, -17.027, +28.011, etc.
    cleaned = re.sub(r'[+-][\d.]+', '', str(peptide_string))
    return cleaned

# Create the new dataframe with required columns
def extract_protein_ids(prots_string):
    """Extract protein IDs from the ProtsAll format"""
    if pd.isna(prots_string):
        return ''
    # Extract protein IDs from format like (sp|Q9H0U3-2|MAGT1_HUMAN,1,13,0,0)
    protein_ids = re.findall(r'\|([A-Z0-9-]+)\|', str(prots_string))
    return ';'.join(protein_ids) if protein_ids else ''

summary_df = pd.DataFrame({
    'scan': mismatch_df['Scan'],
    'DB_search_score': mismatch_df['Score'],
    'precursor_score': mismatch_df['ScoreOther'],
    'DB_proteins': mismatch_df['ProtsAll'].apply(lambda x: extract_protein_ids(x) if 'Annotation' in mismatch_df.columns else ''),
    'precursor_proteins': mismatch_df['ProtsAllOther'].apply(lambda x: extract_protein_ids(x) if 'AnnotationOther' in mismatch_df.columns else ''),
    'precursor_better': mismatch_df['Score'] < mismatch_df['ScoreOther'],
    'charge': mismatch_df['ChargeOther'],
    'peptide': mismatch_df['AnnotationOther'],
    'peptide_demod': mismatch_df['AnnotationOther'].apply(remove_modifications),
    'peptide_length': mismatch_df['AnnotationOther'].apply(remove_modifications).apply(len),
    'reason_mismatch': '',  # Empty for now as requested
    'MinNTermAdd': mismatch_df['MinNTermAdd'],
    'minNTermSubtract': mismatch_df['minNTermSubtract'],
    'MinCTermAdd': mismatch_df['MinCTermAdd'],
    'minCTermSubtract': mismatch_df['minCTermSubtract']
})

# Save to TSV file
summary_df.to_csv('mismatch_summary.tsv', sep='\t', index=False)

print(f"Created mismatch_summary.tsv with {len(summary_df)} rows")
print("\nFirst few rows of the summary:")
summary_df.head()

Created mismatch_summary.tsv with 9020 rows

First few rows of the summary:


,scan,DB_search_score,precursor_score,DB_proteins,precursor_proteins,precursor_better,charge,peptide,peptide_demod,peptide_length,reason_mismatch,MinNTermAdd,minNTermSubtract,MinCTermAdd,minCTermSubtract
0,6352,-25.036301,106.825996,Q86Y38,P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H0...,True,2,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,,0,0,5,21
1,6353,-23.241800,103.260002,Q9H0U3-2;Q9H0U3;A0A087WU53;A0A8I5KUC4;A0A8I5KY...,P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H0...,True,2,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,,0,0,5,21
2,21453,8.635220,95.855103,Q6DT37,Q9BXT2;A6NFR2;A6NP74,True,2,+42.011MM+15.995WSNFFLQEENR,MMWSNFFLQEENR,13,,0,0,0,0
3,6363,-22.113600,89.525597,Q9H0U3-2;Q9H0U3;A0A087WU53;A0A8I5KUC4;A0A8I5KY...,P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H0...,True,2,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,,0,0,5,21
4,5047,-27.427601,87.540901,Q9P1Y6-2;Q9P1Y6-3;Q9P1Y6;A0A0J9YWD5;A0A0J9YX88...,P21815,True,2,DNGYEIYESENGEPRG,DNGYEIYESENGEPRG,16,,18,15,4,1


use quick indexing methods (with progress bar) to read from Human_UP000005640_2025_05_29_with_isoforms_CC_with_decoys.fasta, filter out the peptides if the peptide_demod is not in the fasta file, print the new summary_df head and number of columns 

In [11]:
# from tqdm import tqdm

# # Read FASTA file and create a set of all protein sequences
# print("Reading FASTA file...")
# fasta_sequences = set()

# with open('data/Human_UP000005640_2025_05_29_with_isoforms_CC_with_decoys.fasta', 'r') as f:
#     current_seq = []
#     for line in tqdm(f, desc="Loading FASTA"):
#         line = line.strip()
#         if line.startswith('>'):
#             # Save previous sequence if exists
#             if current_seq:
#                 fasta_sequences.add(''.join(current_seq))
#                 current_seq = []
#         else:
#             current_seq.append(line)
#     # Add the last sequence
#     if current_seq:
#         fasta_sequences.add(''.join(current_seq))

# print(f"Loaded {len(fasta_sequences)} protein sequences from FASTA file")

# # Create a combined string for faster substring searching (optional optimization)
# # For very large FASTA files, searching individual sequences might be faster
# combined_fasta = ''.join(fasta_sequences)

# # Filter peptides that are found in the FASTA file
# print("\nFiltering peptides...")
# peptides_found = []

# for idx, row in tqdm(summary_df.iterrows(), total=len(summary_df), desc="Checking peptides"):
#     peptide_demod = row['peptide_demod']
#     # Check if peptide is a substring of the combined FASTA sequences
#     if peptide_demod in combined_fasta:
#         peptides_found.append(True)
#     else:
#         peptides_found.append(False)

# # Filter the dataframe
# summary_df = summary_df[peptides_found].reset_index(drop=True)

# print(f"\nFiltered summary_df shape: {summary_df.shape}")
# print(f"Number of columns: {len(summary_df.columns)}")
# print("\nFirst few rows:")
# print(summary_df.head())

get the identified_peptide_list as a set of tuples with (peptide_demod, mass_shift, charge), now get them all from matched_df (already read earlier), where the get the peptide demod similar to mismatch df, get the mass shift as +digits or -digits from the peptide modified, and get charge from the df column 'Charge'

In [12]:
from tqdm import tqdm

# Extract peptide information from matched_df
print("Processing matched_df to create identified_peptide_list...")

identified_peptide_list = set()

for idx, row in tqdm(matched_df.iterrows(), total=len(matched_df), desc="Processing matched peptides"):
    # Get demodified peptide
    peptide_demod = remove_modifications(row['Annotation'])
    
    # Extract mass shift from the modified peptide
    mod_matches = re.findall(r'([+-][\d.]+)', str(row['Annotation']))
    mass_shift = mod_matches[0] if mod_matches else '+0'
    
    # Get charge (assuming there's a 'Charge' column in matched_df)
    charge = row.get('Charge', 0)
    
    # Add to set as tuple
    identified_peptide_list.add((peptide_demod, mass_shift, charge))

print(f"\nCreated identified_peptide_list with {len(identified_peptide_list)} unique entries")
print("\nFirst few entries:")
for i, entry in enumerate(list(identified_peptide_list)):
    print(entry)

Processing matched_df to create identified_peptide_list...


Processing matched peptides: 100%|██████████| 9492/9492 [00:00<00:00, 21774.58it/s]


Created identified_peptide_list with 1242 unique entries

First few entries:
('ISIERYLGV', '+0', 0)
('VEPPQYMIDLYNR', '+15.995', 0)
('IVQNGADGSKY', '+0.984', 0)
('EPYAGPQVF', '+0', 0)
('DLLFSIVEEETGKDCAK', '+57.021', 0)
('MNDYKLEEDPVTK', '+0', 0)
('ASETPEDGDPEEDTATALQR', '+0', 0)
('AGFPRPGSLQTFLLR', '+0', 0)
('CDTIYQGFAECLIR', '+57.021', 0)
('DKLAATQKKL', '+0', 0)
('EQSRPDAPITDQDILR', '+0', 0)
('ESCFTQPQGVLSR', '+57.021', 0)
('MELCRSLAL', '+57.021', 0)
('EAFDFVKQRR', '+0', 0)
('DLLFSIVEEETGK', '+0', 0)
('YLSIGHPYFYQR', '+0', 0)
('DTIPNIMFF', '+0', 0)
('VLFKDPVSV', '+0', 0)
('VSIVPTQAVCGLPDR', '+57.021', 0)
('DSLLELSPVER', '+0', 0)
('VVFVYVATR', '+0', 0)
('WAHELLLSFREK', '+0', 0)
('NPIDQCNTL', '+57.021', 0)
('SVLPGVGDAAAAAVAATAVPAVSQAQLGTR', '+0', 0)
('YPGMFIALSK', '+15.995', 0)
('LTQCNVSATLQEPAGTSR', '+57.021', 0)
('ETASPTPLNEV', '+0', 0)
('LQVWSGTEVTAPQGATDR', '+0', 0)
('YEYSLKLLRY', '+0', 0)
('NGVAAFHAFLK', '+0.984', 0)
('AAAGTFKVL', '+0', 0)
('IDREELCMGAIK', '+57.021', 0)
('GIYLDTA

def a fucntion for checking reason mismatch, and filling peptide type,  default is leave blank for mismatch_reason
    miss_3_cleavages ----Peptide has 3+ missed cleavages for trypsin
    lost_3_aa_Nterm--- Peptide lost 3+ AAs from the N-term (prefix) - this is the MinNTermAdd if its larger than 3
    Cterm_non_tryptic --- Peptide C-term (suffix/end) is non-Tryptic (i.e., no K/R at the end)

    for peptide_type column 
        read from peptide_length
        HLA1 -- Peptide length 8-12 = mark as HLA1
        HLA2 -- Peptide length 9-20 = mark as HLA2
        Otherwise mark as OtherLength


In [13]:
def analyze_peptide_mismatches(df):
    """
    Function to analyze peptide mismatches and assign peptide types
    """
    def is_identified(peptide_demod, mass_shift, charge, identified_peptide_list):
        """Check if the peptide is in the identified peptide list"""
        return (peptide_demod, mass_shift, charge) in identified_peptide_list

    def count_missed_cleavages(peptide_seq):
        """Count missed cleavages for trypsin (K/R not at C-terminus)"""
        if pd.isna(peptide_seq) or len(peptide_seq) == 0:
            return 0
        # Count K and R that are not at the end of the sequence
        missed = 0
        for i, aa in enumerate(peptide_seq[:-1]):  # Exclude last position
            if aa in ['K', 'R']:
                missed += 1
        return missed
    
    def is_cterm_tryptic(peptide_seq):
        """Check if C-terminus is tryptic (ends with K or R)"""
        if pd.isna(peptide_seq) or len(peptide_seq) == 0:
            return False
        return peptide_seq[-1] in ['K', 'R']
    
    def check_HLA(length):
        """Assign peptide type based on length"""
        if 8 <= length <= 12:
            return 'HLA1'
        elif 13 <= length <= 24:
            return 'HLA2'
        else:
            return 'Others'
    
    def get_mismatch_reason(row):
        # Check if peptide is identified first
        is_ident = is_identified(row['peptide_demod'], 
                                 row.get('mass_shift', '+0'), 
                                 row.get('Charge', 0), 
                                 identified_peptide_list)
        if is_ident:
            return 'IDENTIFIED'
        """Determine mismatch reason based on criteria - following priority sequence"""
        
        # 1. Check for non-standard modifications first
        standard_mods = [1, 16, 42, 43, -17]
        tolerance = 0.5  # Allow for rounding
        
        # Extract all modifications from peptide string
        mod_matches = re.findall(r'([+-][\d.]+)', str(row['peptide']))
        for mod_str in mod_matches:
            mod_value = float(mod_str)
            is_standard = any(abs(mod_value - std_mod) <= tolerance for std_mod in standard_mods)
            if not is_standard:
                return 'non_standard_modification'
        
        # 2. Check for multiple modifications (2+ modifications)
        mod_count = row['peptide'].count('+') + row['peptide'].count('-')
        if mod_count >= 2:
            return 'multiple_modifications'
        
        # 3. Check if length > 40
        if row['peptide_length'] > 40:
            return 'peptide_length_>_40'
        
        # 4. Check if peptide ends in K/R (is tryptic)
        if is_cterm_tryptic(row['peptide_demod']):
            # 4a. Check for lost 3+ AAs from N-term
            if row['MinNTermAdd'] >= 3:
                return 'lost_3_aa_Nterm'
            
            # 4b. Check for 3+ missed cleavages
            missed_cleavages = count_missed_cleavages(row['peptide_demod'])
            if missed_cleavages >= 3:
                return 'miss_3_cleavages'
            
            else:
                return 'regular_tryptic'
        
        # 5. Check peptide length for HLA classification
        if 8 <= row['peptide_length'] <= 12:
            return 'HLA1'
        elif 13 <= row['peptide_length'] <= 24:
            return 'HLA2'
        
        # 6. If nothing applies
        return 'Others'
    
    # Apply the function to update the dataframe
    df['reason_mismatch'] = df.apply(get_mismatch_reason, axis=1)
    
    return df

# Apply the analysis to the summary dataframe
summary_df = analyze_peptide_mismatches(summary_df)

# Save the updated dataframe
summary_df.to_csv('mismatch_summary.tsv', sep='\t', index=False)

print(f"Updated mismatch_summary.tsv with peptide types and mismatch reasons")
print("\nMismatch reason distribution:")
print(summary_df['reason_mismatch'].value_counts())
print("\nFirst few rows:")
summary_df.head()

Updated mismatch_summary.tsv with peptide types and mismatch reasons

Mismatch reason distribution:
HLA2                         3643
non_standard_modification    1626
IDENTIFIED                   1446
Others                        790
lost_3_aa_Nterm               534
miss_3_cleavages              339
HLA1                          338
multiple_modifications        203
regular_tryptic                52
peptide_length_>_40            49
Name: reason_mismatch, dtype: int64

First few rows:


,scan,DB_search_score,precursor_score,DB_proteins,precursor_proteins,precursor_better,charge,peptide,peptide_demod,peptide_length,reason_mismatch,MinNTermAdd,minNTermSubtract,MinCTermAdd,minCTermSubtract
0,6352,-25.036301,106.825996,Q86Y38,P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H0...,True,2,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,non_standard_modification,0,0,5,21
1,6353,-23.241800,103.260002,Q9H0U3-2;Q9H0U3;A0A087WU53;A0A8I5KUC4;A0A8I5KY...,P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H0...,True,2,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,non_standard_modification,0,0,5,21
2,21453,8.635220,95.855103,Q6DT37,Q9BXT2;A6NFR2;A6NP74,True,2,+42.011MM+15.995WSNFFLQEENR,MMWSNFFLQEENR,13,multiple_modifications,0,0,0,0
3,6363,-22.113600,89.525597,Q9H0U3-2;Q9H0U3;A0A087WU53;A0A8I5KUC4;A0A8I5KY...,P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H0...,True,2,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,non_standard_modification,0,0,5,21
4,5047,-27.427601,87.540901,Q9P1Y6-2;Q9P1Y6-3;Q9P1Y6;A0A0J9YWD5;A0A0J9YX88...,P21815,True,2,DNGYEIYESENGEPRG,DNGYEIYESENGEPRG,16,HLA2,18,15,4,1


In [14]:
# Filter for rows where reason_mismatch is blank and peptide_type is OtherLength
filtered_df = summary_df[(summary_df['reason_mismatch'] == 'Others') ]

# Save the filtered data to a new TSV file
filtered_df.to_csv('not_expected_mismatch.tsv', sep='\t', index=False)

print(f"Created filtered_otherlength_nomismatch.tsv with {len(filtered_df)} rows")
print(f"Total rows in original data: {len(summary_df)}")
print(f"Filtered rows (blank reason_mismatch AND OtherLength): {len(filtered_df)}")
print("\nFirst few rows of filtered data:")
filtered_df.head()

Created filtered_otherlength_nomismatch.tsv with 790 rows
Total rows in original data: 9020
Filtered rows (blank reason_mismatch AND OtherLength): 790

First few rows of filtered data:


,scan,DB_search_score,precursor_score,DB_proteins,precursor_proteins,precursor_better,charge,peptide,peptide_demod,peptide_length,reason_mismatch,MinNTermAdd,minNTermSubtract,MinCTermAdd,minCTermSubtract
238,21260,-59.325001,58.664101,Q5VZ66;A0A590UIU4;A0A590UJ09;A0A590UJH1;A0A590...,Q9BVV8,True,2,LLANTEDPTEMASLDSDEETVFESRNL,LLANTEDPTEMASLDSDEETVFESRNL,27,Others,2,25,1,2
806,21262,-37.429901,42.355400,Q53FD0;A0A669KB15;J3KMY6,Q9BVV8,True,3,LLANTEDPTEMASLDSDEETVFESRNL,LLANTEDPTEMASLDSDEETVFESRNL,27,Others,2,25,1,2
1020,21256,-37.120399,39.083000,Q9UNE7-2;Q9UNE7;H3BS86;H3BUD0,Q9BVV8,True,3,LLANTEDPTEM+15.995ASLDSDEETVFESRNL,LLANTEDPTEMASLDSDEETVFESRNL,27,Others,2,25,1,2
2330,19925,-16.704300,24.587799,Q8N8V2,Q8TF08;A0A0C4DGG2,True,4,TATQIGIEWNLSPVGRVTPKEWKHQ,TATQIGIEWNLSPVGRVTPKEWKHQ,25,Others,19,16,0,0
2502,21257,-37.444199,22.762800,Q14137-2;Q14137;A0A075B729,Q9BVV8,True,3,LLANTEDPTEM+15.995ASLDSDEETVFESRNL,LLANTEDPTEMASLDSDEETVFESRNL,27,Others,2,25,1,2
